# RQ4: Do advanced ensemble models outperform baseline models for fraud detection?

**Research Question:** How do advanced ensemble methods (XGBoost, LightGBM, AdaBoost, Stacking) compare against baseline models in fraud detection accuracy and AUC-ROC?

**Hypothesis:** XGBoost and LightGBM will achieve superior AUC-ROC scores due to gradient boosting's ability to handle complex feature interactions.

**Target Variable:** `is_fraud`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve)
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print('Libraries loaded.')

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
DATA_PATH = '/kaggle/input/financial-transactions-dataset-for-fraud-detection/financial_fraud_detection_dataset.csv'
df = pd.read_csv(DATA_PATH, nrows=50000)

TARGET = 'is_fraud'
DROP_COLS = ['transaction_id','timestamp','sender_account','receiver_account',
             'fraud_type','ip_address','device_hash']
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

df.dropna(inplace=True)
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Split done.')

In [ ]:
# Stacking estimator
estimators = [
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=50, random_state=42, eval_metric='logloss', verbosity=0)),
]
stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3
)

models = {
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM':            LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'Stacking':            stacking,
}

results = []
roc_data = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_data[name] = (fpr, tpr)
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, preds, zero_division=0), 4),
        'F1':        round(f1_score(y_test, preds, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, proba), 4),
    })
    print(f'{name:22s} AUC={results[-1]["AUC-ROC"]}')

df_res = pd.DataFrame(results)
df_res

In [ ]:
df_res.to_csv('table_rq4_ensemble_models.csv', index=False)
print('Table saved.')

In [ ]:
colors = ['#2196F3','#FF9800','#4CAF50','#E91E63','#9C27B0','#795548']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: AUC-ROC bar chart
bars = axes[0].barh(df_res['Model'], df_res['AUC-ROC'],
                    color=colors[:len(df_res)], alpha=0.85, edgecolor='white')
axes[0].set_xlabel('AUC-ROC Score', fontsize=12)
axes[0].set_title('AUC-ROC by Model', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 1.1)
for bar, val in zip(bars, df_res['AUC-ROC']):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[0].xaxis.grid(True, linestyle='--', alpha=0.6)
axes[0].set_axisbelow(True)

# Right: ROC curves
for (name, (fpr, tpr)), color in zip(roc_data.items(), colors):
    auc = df_res[df_res['Model']==name]['AUC-ROC'].values[0]
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curves\nAll Ensemble Models', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8, loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('RQ4: Advanced Ensemble Models for Fraud Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figure_rq4_ensemble_models.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved.')